# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their Croissant `@id` fields for maximum data traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL:**
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata and records interface
dataset = mlc.Dataset(croissant_url)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Explore available record sets, their fields, and respective `@id`s.

Let's print the available record sets (`cr:RecordSet`), their `@id`s, and, for each, corresponding field `@id`s and labels.

In [ ]:
# List record sets and their fields (all referenced by @id)
def get_record_sets(ds):
    rsets = []
    for rset in ds.metadata.record_sets:
        print(f"RecordSet: @id = '{rset.id}', label = '{rset.label}'")
        print(f"  description: {rset.description}")
        print(f"  Fields:")
        for field in rset.fields:
            print(f"    - @id = '{field.id}', label = '{field.label}', dataType = '{field.data_type}', description = {field.description}")
        rsets.append(rset.id)
    return rsets

record_set_ids = get_record_sets(dataset)
print("\nAll discovered record sets:")
print(record_set_ids if record_set_ids else 'No record sets found!')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

The main data table of this dataset is usually labeled as "Main Table" or similar. We'll use its full Croissant `@id` as discovered above.

In [ ]:
# You can customize the chosen record set @id here if needed
main_record_set_id = record_set_ids[0] if record_set_ids else None  # Use the first discovered RecordSet (likely main table)

if main_record_set_id:
    # Load all record sets as DataFrames by @id
    dataframes = dict()
    for rset_id in record_set_ids:
        print(f"Loading records for RecordSet '@id' = {rset_id} ...")
        records = list(dataset.records(record_set=rset_id))
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"  -> Loaded {len(dataframes[rset_id])} records. Columns: {dataframes[rset_id].columns.tolist()}")

    print("\nColumns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the schema.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some typical exploratory steps:

- Select a numeric field (by `@id`) for demonstration (e.g., total peptide count, area, or molecular weight [MW])
- Filter records
- Normalize the field
- Group by a categorical field (`@id`, e.g., sample or protein type) if available

In [ ]:
if main_record_set_id:
    # Auto-detect numeric fields for demonstration
    df = dataframes[main_record_set_id]
    # Pick first numerical column as numeric field, or set manually if the schema is known
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        print("No numeric fields found. Please check the data or refer to the schema.")
    else:
        print(f"Using numeric field '@id' = '{numeric_field}' for EDA.")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        print(f"Filtering records with {numeric_field} > {threshold:.2f} (mean value)...")
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records: {len(filtered_df)} out of {len(df)}")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a group field (categorical) for grouping
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and df[col].nunique() < 20:
                group_field = col
                break
        print(f"Grouping by group field: '{group_field}'" if group_field else "No categorical group field found.")
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df[[numeric_field]].head())
else:
    print("No record sets loaded.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and group means if group field exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if 'filtered_df' in locals() and group_field:
        # Group mean bar plot
        group_means = filtered_df.groupby(group_field)[numeric_field].mean()
        group_means.plot(kind='bar', figsize=(8, 4))
        plt.title(f"Mean {numeric_field} by {group_field} (> mean {numeric_field})")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
- Schema loading and record interface enabled programmatic access to all data tables (`cr:RecordSet`) and their fields/columns by Croissant `@id`.
- We've viewed and normalized a representative numeric field, filtering and grouping to reveal trends.
- This workflow is fully compatible with changes to the Croissant schema, since all entity references use persistent `@id`s.

**Further analysis:**
- To refine exploration, consult the Croissant metadata for additional field semantics or run domain-specific queries.
- For more information, see: [https://mlcommons.github.io/croissant/python](https://mlcommons.github.io/croissant/python)
